# ADMM demo

this notebook:

1. clones the repository and installs dependencies
2. downloads checkpoints from Hugging Face when the selected model needs one
3. downloads a custom `.zip` dataset from Google Drive
4. runs `inference.py` and saves reconstructions
5. visualizes `original | lensless | reconstruction` samples
6. runs `calculate_metrics.py` when `lensed/` ground truth is present

dataset structure inside the zip:

```md
any_root_name/
- lensless/
-- sample_1.png
- masks/
-- sample_1.npy
- lensed/ (optional)
-- sample_1.png
```

reconstructions will be stored in `data/saved/demo_reconstructions/test/`

In [ ]:
REPO_URL = "https://github.com/akhasanovv/admm-implementation.git"
BRANCH = None

# paste google drive link to dataset.zip here
DATASET_ZIP_URL = "https://drive.google.com/file/d/1yAUQ_qRNcLmP1CstU09IfFidTK1jFlRf/view?usp=sharing"
OUTPUT_NAME = "demo_reconstructions"

# possible values: admm, le_admm, pre4_leadmm5_post4, pre8_leadmm5, leadmm5_post8
MODEL_NAME = "le_admm"

# hf repo with trained models
CHECKPOINT_REPO_ID = "akhasanovv/admm-lensless-checkpoints"
CHECKPOINT_FILES = {
    "le_admm": "le_admm20.pth",
    "pre4_leadmm5_post4": "pre4_leadmm5_post4.pth",
    "pre8_leadmm5": "pre8_leadmm5.pth",
    "leadmm5_post8": "leadmm5_post8.pth",
}

N_VIS = 3


### clone the repo and install dependencies


In [ ]:
!git clone https://github.com/akhasanovv/admm-implementation.git
%cd admm-implementation
!pip install -r requirements.txt

### download checkpoint (for pretrained models)

`admm` has fixed parameters, so it does not need a checkpoint. for other models 
this cell downloads the selected checkpoint from hf repo


In [2]:
from huggingface_hub import hf_hub_download

checkpoint_path = None
if MODEL_NAME in CHECKPOINT_FILES:
    checkpoint_path = hf_hub_download(
        repo_id=CHECKPOINT_REPO_ID,
        filename=CHECKPOINT_FILES[MODEL_NAME],
        repo_type="model",
    )
    print(f"checkpoint: {checkpoint_path}")
elif MODEL_NAME == "admm":
    print("admm doesn't need a checkpoint")
else:
    raise ValueError(f"unknown MODEL_NAME: {MODEL_NAME}")


/Users/aijdwr/admm-implementation/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


checkpoint: /Users/aijdwr/.cache/huggingface/hub/models--akhasanovv--admm-lensless-checkpoints/snapshots/913ea6d2f17cc581f1a05c40fc423c35574215cb/le_admm20.pth


### download custom dataset

set `DATASET_ZIP_URL` in the first cell (this can be a normal google drive share link). 
the code finds the directory that contains `lensless/` and `masks/` after unzip


In [5]:
import zipfile
import shutil
from pathlib import Path

import gdown

if not DATASET_ZIP_URL:
    raise ValueError("set DATASET_ZIP_URL")

ZIP_PATH = Path("/content/custom_dataset.zip")
DATA_ROOT = Path("/content/custom_dataset")

if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

gdown.download(url=DATASET_ZIP_URL, output=str(ZIP_PATH), quiet=False, fuzzy=True)
if not ZIP_PATH.exists():
    raise FileNotFoundError(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(DATA_ROOT)

def find_dataset_dir(root):
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for path in candidates:
        if (path / "lensless").is_dir() and (path / "masks").is_dir():
            return path
    raise FileNotFoundError("no directory with lensless/ and masks/ was found")

DATASET_DIR = find_dataset_dir(DATA_ROOT)
print(f"dataset: {DATASET_DIR}")
print(f"has ground truth: {(DATASET_DIR / 'lensed').is_dir()}")


OSError: [Errno 30] Read-only file system: '/content'

### inference

calls `inference.py`, reconstructions are saved to `data/saved/<OUTPUT_NAME>/test/`


In [ ]:
import sys
import subprocess
from pathlib import Path

import torch

PROJECT_DIR = Path("/content/admm-implementation")

print(f"dataset: {DATASET_DIR}")
print(f"lensless: {len(list((DATASET_DIR / 'lensless').glob('*')))}")
print(f"masks: {len(list((DATASET_DIR / 'masks').glob('*')))}")
print(f"lensed: {len(list((DATASET_DIR / 'lensed').glob('*')))}")

if checkpoint_path is not None:
    checkpoint_path = Path(checkpoint_path)
    print(f"checkpoint: {checkpoint_path}")
    print(f"checkpoint exists: {checkpoint_path.exists()}")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if isinstance(checkpoint, dict) and "config" in checkpoint:
        print(f"checkpoint model: {checkpoint['config']['model']}")

cmd = [
    sys.executable,
    "inference.py",
    f"model={MODEL_NAME}",
    "datasets=custom_dir_eval",
    f"datasets.test.data_dir={DATASET_DIR}",
    "dataloader=lensless",
    "dataloader.batch_size=1",
    "dataloader.num_workers=0",
    "metrics=reconstruction_no_lpips",
    f"inferencer.save_path={OUTPUT_NAME}",
]
if checkpoint_path is not None:
    cmd.append(f"inferencer.from_pretrained={checkpoint_path}")

print("command:", " ".join(map(str, cmd)))

result = subprocess.run(
    cmd,
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
)

print("stdout:")
print(result.stdout)
print("stderr:")
print(result.stderr)
result.check_returncode()

PRED_DIR = PROJECT_DIR / "data" / "saved" / OUTPUT_NAME / "test"
print(f"reconstructions: {PRED_DIR}")


### visualization

when `lensed/` exists, each row is `original | lensless | reconstruction`. otherwise -- `lensless | reconstruction`


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def image_index(path):
    path = Path(path)
    if not path.is_dir():
        return {}
    return {
        file.stem: file
        for file in sorted(path.iterdir())
        if file.suffix.lower() in IMAGE_EXTENSIONS
    }

lensless_files = image_index(DATASET_DIR / "lensless")
lensed_files = image_index(DATASET_DIR / "lensed")
pred_files = image_index(PRED_DIR)

sample_ids = sorted(pred_files.keys())[:N_VIS]
if not sample_ids:
    raise FileNotFoundError(f"no reconstructed images in {PRED_DIR}")

has_gt = bool(lensed_files)
cols = 3 if has_gt else 2
fig, axes = plt.subplots(len(sample_ids), cols, figsize=(4 * cols, 3 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = [axes]

for row, image_id in enumerate(sample_ids):
    row_axes = axes[row]
    col = 0

    if has_gt:
        row_axes[col].imshow(Image.open(lensed_files[image_id]).convert("RGB"))
        row_axes[col].set_title("original")
        row_axes[col].axis("off")
        col += 1

    row_axes[col].imshow(Image.open(lensless_files[image_id]).convert("RGB"))
    row_axes[col].set_title("lensless")
    row_axes[col].axis("off")
    col += 1

    row_axes[col].imshow(Image.open(pred_files[image_id]).convert("RGB"))
    row_axes[col].set_title("reconstruction")
    row_axes[col].axis("off")

plt.tight_layout()


### metrics

if `lensed/` exists, this runs `calculate_metrics.py` and outputs MSE, PSNR, SSIM, and LPIPS


In [ ]:
if (DATASET_DIR / "lensed").is_dir():
    metric_cmd = [
        sys.executable,
        "calculate_metrics.py",
        "--gt-dir",
        str(DATASET_DIR / "lensed"),
        "--pred-dir",
        str(PRED_DIR),
        "--lpips",
    ]
    subprocess.run(metric_cmd, cwd=PROJECT_DIR, check=True)
else:
    print("no lensed/ directory found")
